# M.I.N.E.R.V.A. v2.3 — From-Scratch Colab Run

**Misinformation Investigation through Networked Embeddings for Rumor Verification and Awareness**
FEU Institute of Technology IT thesis 2026

This notebook runs the full pipeline end-to-end from a fresh Colab session.

**v2.3 changes (since v2.2):**
- Curation now produces a **POOL** of 500 cards (not a fixed 56-card deck)
- New script 28 draws a **per-user 56-card deck** deterministically from the pool
- Each Filipino SHS student gets a different deck — supports educational integrity and independent per-decision data for the thesis evaluation
- Same `user_id` always produces the same deck (reproducible, resumable)

**Pipeline:**
```
[Setup -> tests] -> [Train detectors -> train GPT-2] -> [Generate 3000 posts -> score]
              -> [v2.x explain layer] -> [500-card POOL] -> [Per-user deck draw]
```

## 1. Configuration

Edit this cell once. Everything else uses these values.

In [ ]:
# Repo
REPO_URL    = "https://github.com/robertgeraldsenasin/MINERVA.git"
BRANCH_NAME = "upgrade/minerva-election-theme"
WORKDIR     = "/content"
REPO_DIR    = f"{WORKDIR}/MINERVA"

# Drive (optional - set to False if your account times out)
USE_DRIVE        = True
DRIVE_INPUT_DIR  = "/content/drive/MyDrive/MINERVA_outputs_revised"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/MINERVA_v2_outputs"

# Pipeline scale (v2.3)
N_GENERATE_FAKE  = 1500   # GPT-2 samples per class (was 500 in v2.2)
N_GENERATE_REAL  = 2500   # v2.4: bumped from 1500 — REAL has higher attrition
POOL_SIZE        = 500    # total POOL of curated cards (was 56 fixed deck in v2.2)
DAYS_IN_DECK     = 7      # number of in-game days per player
CARDS_PER_DAY    = 8      # cards shown per day per player
MIN_CREDIBLE_PER_DAY = 3  # quota of REAL cards per day (Modirrousta-Galian & Higham 2023)

print(f"Repo:    {REPO_URL}")
print(f"Branch:  {BRANCH_NAME}")
print(f"Workdir: {REPO_DIR}")
print(f"Drive:   {'enabled' if USE_DRIVE else 'DISABLED'}")
print(f"Pool size: {POOL_SIZE} cards (each player draws {DAYS_IN_DECK*CARDS_PER_DAY})")
print(f"Capacity: ~{POOL_SIZE // (DAYS_IN_DECK*CARDS_PER_DAY)} unique decks possible")

## 2. Mount Drive (gracefully — continues if it fails)

If Drive mount times out, the notebook falls back to runtime-only mode. **Setting `USE_DRIVE = False` in cell 1 skips this entirely.**

In [ ]:
DRIVE_MOUNT_OK = False

if USE_DRIVE:
    from google.colab import drive
    import os
    print("Hosted Colab runtime:", os.path.exists("/var/colab/hostname"))
    print("Attempting to mount Drive (5-min timeout)...")
    try:
        drive.mount("/content/drive", force_remount=True, timeout_ms=300000)
        DRIVE_MOUNT_OK = True
        print("✓ Drive mounted")
    except Exception as e:
        print(f"✗ Drive mount failed: {e!r}")
        print("  Continuing in runtime-only mode.")
        print("  At the end you'll get a downloadable zip instead of Drive save.")
else:
    print("Drive disabled by configuration (USE_DRIVE=False).")
    print("All outputs will be downloaded as a zip at the end.")

print(f"\nDRIVE_MOUNT_OK = {DRIVE_MOUNT_OK}")

## 3. Clone the upgrade branch from GitHub

In [ ]:
import os, shutil

if os.path.exists(REPO_DIR):
    print(f"Removing existing {REPO_DIR}...")
    shutil.rmtree(REPO_DIR)

!git clone --branch "$BRANCH_NAME" "$REPO_URL" "$REPO_DIR"
%cd $REPO_DIR
!git log -1 --oneline
print()
print("Top-level files:")
!ls

## 4. Verify v2.x files arrived

In [ ]:
from pathlib import Path

REQUIRED = [
    "scripts/minerva_indicators.py",
    "scripts/minerva_response_bank.py",
    "scripts/minerva_schemas.py",
    "scripts/minerva_candidates.py",
    "scripts/minerva_filters.py",
    "scripts/13_score_generated_with_qlattice.py",
    "scripts/18_verdict_explain.py",
    "scripts/22_pseudonymize_entities.py",
    "scripts/23_enforce_election_theme.py",
    "scripts/24_curate_teaching_cards.py",
    "scripts/25_build_candidate_scenarios.py",
    "scripts/26_faithfulness_audit.py",
    "scripts/27_response_bank_versioning.py",
    "scripts/28_draw_user_deck.py",
    "templates/candidate_profiles_three_candidates.json",
    "templates/teaching_response_bank_v1.json",
    "tests/test_indicators.py",
    "tests/test_filters.py",
]
missing = [p for p in REQUIRED if not Path(p).exists()]
if missing:
    print(f"✗ MISSING {len(missing)} files:")
    for m in missing:
        print(f"  {m}")
    raise RuntimeError("Cannot proceed with missing files")
print(f"✓ All {len(REQUIRED)} v2.x files present")

## 5. Install dependencies

In [ ]:
!python -m pip uninstall -y peft trl
!python -m pip install --upgrade -q pip setuptools wheel
!python -m pip install -q -r requirements.txt
!python -m pip install -q pydantic pytest
!python -m pip install -q networkx spacy feyn torch-geometric

import importlib.metadata as md
for pkg in ["transformers", "torch", "pydantic", "feyn", "spacy"]:
    try:
        print(f"  {pkg}: {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"  {pkg}: NOT INSTALLED")

## 6. spaCy model + working folders

In [ ]:
!python -m spacy download en_core_web_sm

from pathlib import Path
for d in ["data", "models", "generated", "generated/decks", "reports", "outputs"]:
    Path(d).mkdir(parents=True, exist_ok=True)
print("Created/verified: data, models, generated, generated/decks, reports, outputs")

## 7. Sanity check — run all 38 unit tests

In [ ]:
!python -m pytest tests/ -v

## 8. Inspect v2.x building blocks

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(REPO_DIR, "scripts"))

from minerva_response_bank import bank_stats, BANK_VERSION
from minerva_candidates import REGISTRY
import json

print("=== Response Bank ===")
print(json.dumps(bank_stats(), indent=2))
print()
print("=== Candidate Registry ===")
for code_, c in REGISTRY.items():
    print(f"\n{code_} — {c.name} ({c.archetype})")
    print(f"  Region: {c.region}, Party: {c.party_acronym}")
    print(f"  Slogan: {c.platform_slogan}")

## 9. Skip-or-train decision

In [ ]:
import os, shutil
from pathlib import Path

SKIP_TRAINING = False
SKIP_GENERATION = False

if DRIVE_MOUNT_OK and os.path.exists(DRIVE_INPUT_DIR):
    print(f"Found Drive input dir: {DRIVE_INPUT_DIR}")

    src_models = Path(DRIVE_INPUT_DIR) / "models"
    if src_models.exists():
        print(f"Copying models from {src_models}...")
        shutil.copytree(src_models, "models", dirs_exist_ok=True)
        SKIP_TRAINING = True
        print("✓ Models copied → SKIP_TRAINING=True")

    src_gen = Path(DRIVE_INPUT_DIR) / "generated"
    if src_gen.exists():
        shutil.copytree(src_gen, "generated", dirs_exist_ok=True)
        if Path("generated/gpt2_synthetic_samples_fake.jsonl").exists() and \
           Path("generated/gpt2_synthetic_samples_real.jsonl").exists():
            SKIP_GENERATION = True
            print("✓ Generated samples found → SKIP_GENERATION=True")

    src_data = Path(DRIVE_INPUT_DIR) / "data"
    if src_data.exists():
        shutil.copytree(src_data, "data", dirs_exist_ok=True)
        print(f"✓ Data copied")
else:
    if not DRIVE_MOUNT_OK:
        print("Drive not mounted → cells 10-13 will run training from scratch (~3 hours).")
    else:
        print(f"Drive mounted but {DRIVE_INPUT_DIR} not found → training from scratch.")

print(f"\nSKIP_TRAINING   = {SKIP_TRAINING}")
print(f"SKIP_GENERATION = {SKIP_GENERATION}")

## 10. Train detectors

⚠️ Multi-hour GPU training. Skipped if models came from Drive.

In [ ]:
if SKIP_TRAINING:
    print("⏭️  Skipping detector training (models loaded from Drive)")
else:
    !python scripts/01_download_dataset.py
    !python scripts/02_prepare_dataset.py
    !python scripts/03_split_dataset.py
    !python scripts/17_run_5seeds_detectors.py --run_id RUN1
    !python scripts/export_best_detectors_by_val.py --run_id RUN1
    !python scripts/06_extract_features.py
    !python scripts/07_train_random_forest.py
    !python scripts/08_train_qlattice.py
    !python scripts/09_train_degnn.py
    print("\n✓ Detectors trained")

## 11. Train GPT-2 generator

In [ ]:
if SKIP_TRAINING:
    print("⏭️  Skipping GPT-2 fine-tune")
else:
    !python scripts/10_prepare_gpt2MINERVA.py
    !python scripts/11_train_gpt2MINERVA.py
    print("\n✓ GPT-2 fine-tuned")

## 12. Patch + run GPT-2 generation (v2.3 — 3000 samples for 500-card pool)

In [ ]:
if SKIP_GENERATION:
    print("⏭️  Skipping GPT-2 generation (samples loaded from Drive)")
else:
    # Apply tokenizer patch
    from pathlib import Path
    p = Path("scripts/12_generate_gpt2MINERVA.py")
    text = p.read_text(encoding="utf-8")
    old1 = '    gen_tok = AutoTokenizer.from_pretrained(str(paths.gpt2_dir), use_fast=True)\n'
    new1 = (
        '    gen_tok = AutoTokenizer.from_pretrained(str(paths.gpt2_dir), use_fast=True)\n'
        '    gen_tok.padding_side = "left"\n'
        '    if gen_tok.pad_token is None:\n'
        '        gen_tok.pad_token = gen_tok.eos_token\n'
    )
    if old1 in text and new1 not in text:
        text = text.replace(old1, new1, 1)
    old2 = '    gen_tok.add_special_tokens(special)\n'
    new2 = (
        '    gen_tok.add_special_tokens(special)\n'
        '    if gen_tok.pad_token is None:\n'
        '        gen_tok.pad_token = gen_tok.eos_token\n'
        '    gen_mdl.config.pad_token_id = gen_tok.pad_token_id\n'
    )
    if old2 in text and new2 not in text:
        text = text.replace(old2, new2, 1)
    text = text.replace('        pad_token_id=gen_tok.eos_token_id,\n',
                        '        pad_token_id=gen_tok.pad_token_id,\n', 1)
    p.write_text(text, encoding="utf-8")
    print(f"✓ Patched {p}")

    !python scripts/12_generate_gpt2MINERVA.py {N_GENERATE_FAKE} fake 0.70 120 --accept_mode ensemble3 --out generated/gpt2_synthetic_samples_fake.jsonl
    !python scripts/12_generate_gpt2MINERVA.py {N_GENERATE_REAL} real 0.70 120 --accept_mode ensemble3 --out generated/gpt2_synthetic_samples_real.jsonl
    print("\n✓ Generated synthetic posts")

## 13. Score generated posts with QLattice (v2.1 fixed)

In [ ]:
!python scripts/13_score_generated_with_qlattice.py \
    --in_file  generated/gpt2_synthetic_samples_fake.jsonl \
    --out_file generated/gpt2_synthetic_final_fake.jsonl \
    --rejection_log reports/score_rejection_log_fake.jsonl

!python scripts/13_score_generated_with_qlattice.py \
    --in_file  generated/gpt2_synthetic_samples_real.jsonl \
    --out_file generated/gpt2_synthetic_final_real.jsonl \
    --rejection_log reports/score_rejection_log_real.jsonl

!cat generated/gpt2_synthetic_final_fake.jsonl generated/gpt2_synthetic_final_real.jsonl > generated/gpt2_synthetic_final_both.jsonl

import os
size = os.path.getsize('generated/gpt2_synthetic_final_both.jsonl')
print(f"\nFinal both size: {size:,} bytes")
assert size > 0, "ERROR: scoring produced empty output"
!wc -l generated/gpt2_synthetic_final_both.jsonl

## 14. Script 18 — Verdict + explanation (v2.2 with alignment guard)

In [ ]:
!python scripts/18_verdict_explain.py \
    --in_file   generated/gpt2_synthetic_final_both.jsonl \
    --out_file  generated/unity_cards.json \
    --audit_out reports/audit_18.jsonl \
    --seed 1729

## 15. Script 21 — Balance unity cards (v2.3 — 500-card pool)

In [ ]:
!python scripts/21_balance_unity_cards.py \
    --in_file  generated/unity_cards.json \
    --out_file generated/unity_cards_balanced.json \
    --target_total {POOL_SIZE} \
    --report_out reports/balance_report.json

## 16. Script 22 — Pseudonymise to 3 candidates (v2.2 with collapse)

In [ ]:
!python scripts/22_pseudonymize_entities.py \
    --in_file  generated/unity_cards_balanced.json \
    --out_file generated/unity_cards_pseudonymized.json \
    --re_explain --seed 1729

## 17. Script 23 — Enforce electoral theme (v2.2 with sports/showbiz negatives)

In [ ]:
!python scripts/23_enforce_election_theme.py \
    --in_file  generated/unity_cards_pseudonymized.json \
    --out_file generated/unity_cards_themed.json \
    --theme_threshold 0.55 \
    --report_out reports/theme_filter_report.json \
    --rejection_log reports/theme_rejection_log.jsonl

## 18. Script 24 — Curate POOL (v2.3 — produces 500-card pool, not single deck)

In [ ]:
!python scripts/24_curate_teaching_cards.py \
    --in_file  generated/unity_cards_themed.json \
    --out_file generated/unity_cards_pool.json \
    --reject_out generated/pool_rejected.json \
    --report_out reports/pool_curation_report.json \
    --target_pool_size {POOL_SIZE} \
    --days {DAYS_IN_DECK} --cards_per_day {CARDS_PER_DAY} \
    --min_credible_per_day {MIN_CREDIBLE_PER_DAY} \
    --seed 1729

## 18b. Draw per-user decks from the pool (v2.3 dynamic content)

Each Filipino SHS student gets their own deck drawn deterministically from the pool. Same `user_id` always produces the same deck (so the user can resume; researchers can reproduce). Different `user_ids` produce different decks (so students can't share answers and per-decision data is independent observations).

For your initial pilot, draw 5 demo decks. For the full SHS rollout, replace the `--user_ids` argument with a comma-separated list of actual student IDs or a path to a text file with one ID per line.

In [ ]:
# Draw 5 demo decks from the pool to verify dynamic content works
!python scripts/28_draw_user_deck.py \
    --pool_file generated/unity_cards_pool.json \
    --out_dir generated/decks \
    --user_ids demo \
    --report_out reports/draw_report.json

# Show the report so we can see pairwise overlap
import json
report = json.load(open('reports/draw_report.json'))
print(f"Drew {report['n_users_drawn']} decks of size {report['deck_shape']['deck_size']}")
print(f"Pool size: {report['pool_size']} cards, hash: {report['pool_hash']}")
if report.get('pairwise_overlap'):
    print(f"Pairwise overlap: mean {report['pairwise_overlap']['mean_overlap_pct']}%, max {report['pairwise_overlap']['max_overlap_pct']}%")
print()
print("Per-deck summary:")
for d in report['decks']:
    print(f"  {d['user_id']}: {d['deck_size']} cards, verdicts {d['verdicts']}, candidates {d['candidates']}")

## 19. Script 25 — VERIdex candidate scenarios

Reads from the POOL (so VERIdex profiles reflect the full breadth of cards available, not just one player's deck).

In [ ]:
!python scripts/25_build_candidate_scenarios.py \
    --story_cards generated/unity_cards_pool.json \
    --out_file    generated/candidate_scenarios.json

## 20. Script 26 — Faithfulness audit on the pool (must be ≥95%)

In [ ]:
!python scripts/26_faithfulness_audit.py \
    --in_file       generated/unity_cards_pool.json \
    --report_out    reports/faithfulness_audit_report.json \
    --failures_out  reports/faithfulness_failures.jsonl

## 21. Script 27 — Stamp bank version on the pool

In [ ]:
!python scripts/27_response_bank_versioning.py stamp \
    --in_file  generated/unity_cards_pool.json \
    --out_file generated/unity_cards_pool_stamped.json

!python scripts/27_response_bank_versioning.py export \
    --out_file templates/teaching_response_bank_v1_export.json

## 22. Final reports dashboard

In [ ]:
import json, glob

for fp in sorted(glob.glob('reports/*.json')):
    print(f'\n{"="*60}')
    print(fp)
    print('='*60)
    print(json.dumps(json.load(open(fp)), indent=2)[:2000])

## 23. Sample cards from a player deck

Loads the demo player's deck (drawn dynamically from the pool) and shows representative cards. Different `user_id` would give a different selection.

In [ ]:
import json

deck_doc = json.load(open('generated/decks/deck_demo_user_01.json'))
deck = deck_doc['cards']
metadata = deck_doc['_metadata']

print(f"Deck for user '{metadata['user_id']}' (drawn from pool {metadata['pool_hash']})")
print(f"Deck size: {len(deck)} cards across {metadata['days']} days")
print(f"Verdict distribution: {metadata['verdict_dist']}")
print(f"Candidate distribution: {metadata['candidate_dist']}")
print()

if not deck:
    print("ERROR: empty deck. Check pool size and curation report.")
else:
    seen = set()
    for c in deck:
        key = (c['verdict'], c['candidate'])
        if key in seen:
            continue
        seen.add(key)
        print(f"--- Day {c['day']} | {c['verdict']} | {c['candidate']} | tier={c['explanation']['tier']} ---")
        print(f"Text: {c['text'][:150]}...")
        print(f"Fired: {c['fired_indicators']}")
        print(f"Summary: {c['explanation']['summary'][:300]}")
        print()

## 24. Save outputs

Saves the **pool** (not a single deck) and any drawn decks. Unity ships with the pool and re-runs the draw at runtime per-user.

In [ ]:
if DRIVE_MOUNT_OK:
    import shutil
    from pathlib import Path

    drive_out = Path(DRIVE_OUTPUT_DIR)
    if drive_out.exists():
        shutil.rmtree(drive_out)

    KEEP = [
        "generated", "reports",
        "models/roberta_finetuned",
        "models/distilbert_multilingual_finetuned",
        "models/gpt2_tagalog_finetuned",
        "models/pca_roberta.joblib",
        "models/pca_distilbert.joblib",
        "models/degnn.pt",
        "models/degnn_artifacts.joblib",
        "models/qlattice_equation.txt",
        "models/best_qlattice.json",
        "models/qlattice",
        "templates",
    ]
    IGNORE = shutil.ignore_patterns(
        "checkpoint-*", "optimizer.pt", "scheduler.pt",
        "trainer_state.json", "rng_state.pth", "*.tmp",
    )
    for rel in KEEP:
        src = Path(rel)
        if not src.exists():
            print(f"[skip] {rel}")
            continue
        dst = drive_out / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        if src.is_dir():
            shutil.copytree(src, dst, dirs_exist_ok=True, ignore=IGNORE)
        else:
            shutil.copy2(src, dst)
        print(f"[ok] {rel} → {dst}")
    print(f"\nDrive export complete: {drive_out}")
    print(f"\nFor Unity, copy these to your Android project:")
    print(f"  {drive_out}/generated/unity_cards_pool.json    → ship with APK")
    print(f"  {drive_out}/generated/candidate_scenarios.json → VERIdex profiles")
    print(f"  Port scripts/28_draw_user_deck.py to C# for runtime per-user draw")
else:
    import shutil, os
    from google.colab import files
    out_dir = '/tmp/minerva_v2_deliverables'
    os.makedirs(out_dir, exist_ok=True)
    for f in ['generated/unity_cards_pool.json',
              'generated/unity_cards_pool_stamped.json',
              'generated/candidate_scenarios.json',
              'generated/unity_cards_themed.json']:
        if os.path.exists(f):
            shutil.copy(f, f'{out_dir}/{os.path.basename(f)}')
    if os.path.exists('generated/decks'):
        shutil.copytree('generated/decks', f'{out_dir}/decks', dirs_exist_ok=True)
    if os.path.exists('reports'):
        shutil.copytree('reports', f'{out_dir}/reports', dirs_exist_ok=True)
    if os.path.exists('templates'):
        shutil.copytree('templates', f'{out_dir}/templates', dirs_exist_ok=True)
    shutil.make_archive('/tmp/minerva_v2_deliverables', 'zip', out_dir)
    print('Downloading deliverables zip to your laptop...')
    files.download('/tmp/minerva_v2_deliverables.zip')

---

## Appendix — How dynamic content works in v2.3

| Concept | Pre-v2.3 | v2.3 |
|---|---|---|
| GPT-2 generation | 1,000 raw posts | 3,000 raw posts |
| Curation output | 56-card fixed deck | **500-card pool** |
| Each player sees | Same 56 cards in same order | **56 cards drawn from the pool by user_id** |
| Pairwise deck overlap | 100% | ~11% (with 500-card pool) |
| Max distinct decks | 1 | **~9 fully non-overlapping**, hundreds with mild overlap |
| Reproducibility | Deck = seed | Deck = (user_id, pool_hash) |
| Unity runtime | Loads `story_cards.json` directly | Loads `unity_cards_pool.json` + draws per-user using the C# port of `28_draw_user_deck.py` |